<a href="https://colab.research.google.com/github/nazaninbondarian/MachineLearning2024/blob/main/First_Step/TD3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import copy
import random
from collections import deque
# Plotting rewards
import math as mt

In [2]:

# Define the Actor (Policy) Network

class TD3Agent:
	def __init__(self ,

		gama ,
		n_td_step ,
		eps ,
		buffer_size ,
		batch_size,
		state_dim,
		action_dim,
		max_action,
		TAU,
		ACTOR_LR,
		CRITIC_LR,
		POLICY_UPDATE_FREQ,
		NOISE_STD,
		NOISE_CLIP,
		):

		self.gama = torch.tensor([gama])

		self.n_td_step = int(n_td_step)
		self.eps = eps
		self.buffer_size = buffer_size
		self.batch_size = batch_size
		self.TAU = TAU
		self.ACTOR_LR = ACTOR_LR
		self.CRITIC_LR = CRITIC_LR
		self.POLICY_UPDATE_FREQ = POLICY_UPDATE_FREQ
		self.NOISE_STD = NOISE_STD
		self.NOISE_CLIP = NOISE_CLIP

		self.state_dim = state_dim

		self.losses = []

		# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
		self.device = torch.device("cpu")

		self.action_dim = action_dim
		self.max_action = max_action
		self.average_reward = 0
		self.baseline = 0

		self.buffer = deque(maxlen = self.buffer_size)

		self.total_it = 0

		self.counter = 0


	def act(self, state , expolartiy = True):

		state = torch.FloatTensor(state.reshape(1, -1)).to(self.device)

		if expolartiy:

			if random.random() < 0.3:
				ret = np.random.uniform(-self.max_action, self.max_action, size=self.action_dim)
			else:
				ret = self.actor(state).cpu().data.numpy().flatten()

		else:
			ret = self.actor(state).cpu().data.numpy().flatten()

		return ret




	def buffer_add(self , state , action , reward , new_state , done):

		self.buffer.append(
			(
				torch.tensor(
					np.concatenate((state, action))
				, dtype=torch.float32),

				torch.tensor([reward] , dtype=torch.float32 ),

				torch.tensor( new_state , dtype=torch.float32),

				self.gama * ( 1.0 - int(done) ),
			)
		)


	def train(self):

		if len(self.buffer) < self.batch_size:
			return True

		self.total_it += 1

		sample_batch = random.sample(self.buffer, self.batch_size)

		pairs, reward , new_state , done_scale = zip(*sample_batch)

		pairs = torch.stack(pairs)
		reward = torch.stack(reward)
		new_state = torch.stack(new_state)
		done_scale = torch.stack(done_scale)


		noise = (
			torch.randn(self.batch_size, self.action_dim) * self.NOISE_STD
		).clamp(-self.NOISE_CLIP, self.NOISE_CLIP)

		# Target policy smoothing
		next_actions = (self.actor_target(new_state) + noise).clamp(
			-self.max_action, self.max_action
		)

		last_pair = torch.cat( (new_state , next_actions), dim=1)

		with torch.no_grad():

			target_Q1 = self.critic_target_1(last_pair)
			target_Q2 = self.critic_target_2(last_pair)
			values = torch.min(target_Q1, target_Q2)
			target = reward + done_scale * values

		self.critic_optimizer_1.zero_grad()
		output_Q1 = self.critic_1(pairs)
		loss_Q1 = nn.MSELoss()(output_Q1, target)
		loss_Q1.backward()
		self.critic_optimizer_1.step()

		self.critic_optimizer_2.zero_grad()
		output_Q2 = self.critic_2(pairs)
		loss_Q2 = nn.MSELoss()(output_Q2, target)
		loss_Q2.backward()
		self.critic_optimizer_2.step()

		if self.total_it % self.POLICY_UPDATE_FREQ == 0:

			states = pairs[:, :self.state_dim]

			act_pair = torch.cat((states, self.actor(states)), dim=1)
			actor_loss = -self.critic_1( act_pair).mean()
			self.actor_optimizer.zero_grad()
			actor_loss.backward()
			self.actor_optimizer.step()

			for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
				target_param.data.copy_(self.TAU * param.data + (1 - self.TAU) * target_param.data)

			for param, target_param in zip(self.critic_1.parameters(), self.critic_target_1.parameters()):
				target_param.data.copy_(self.TAU * param.data + (1 - self.TAU) * target_param.data)

			for param, target_param in zip(self.critic_2.parameters(), self.critic_target_2.parameters()):
				target_param.data.copy_(self.TAU * param.data + (1 - self.TAU) * target_param.data)
